# CatBoost Taxi Destination Notebook

Standard three-mode CatBoost workflow: validation-ranked hybrid tuning,
selected-model validation diagnostics, and final iteration calibration plus
complete-data fit/submission.


## Cached Data Paths

Load the CatBoost-specific schema-v2 cached preprocessing dataset and write
standard Kaggle outputs.


In [ ]:
import os
from pathlib import Path

PREPROCESSED_DATA_DIR = Path(
    os.environ.get(
        "PREPROCESSED_DATA_DIR",
        (
            "/kaggle/input/datasets/louishogge/"
            "preprocessed-taxi-dataset-catboost/data"
        ),
    )
)
OUTPUT_DIR = Path("/kaggle/working")
SUBMISSION_DIR = OUTPUT_DIR / "submission"
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Preprocessed data directory:", PREPROCESSED_DATA_DIR)
print("Output directory:", OUTPUT_DIR.resolve())
print("Submission directory:", SUBMISSION_DIR.resolve())
print("Artifact directory:", ARTIFACT_DIR.resolve())


## Imports

Import the seeded training, evaluation, plotting, and serialization stack.
CatBoost GPU training remains explicitly recorded as nondeterministic.


In [ ]:
import gc
import json
import random
import tempfile
import time
from copy import deepcopy

import catboost
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from catboost.utils import get_gpu_device_count
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit


## Hyperparameters

Enable at most one expensive mode. Tuning creates unreviewed in-memory evidence;
validation refit and final fit use the reviewed target-specific configuration.


In [ ]:
SEED = 42
MODEL_NAME = "CatBoost"
MODEL_SLUG = "catboost"

RUN_HYPERPARAMETER_TUNING = False
RUN_VALIDATION_REFIT = False
RUN_FINAL_FIT = True

MAX_ITERATIONS = 20_000
EARLY_STOPPING_ROUNDS = 100
SEARCH_STOP_FRACTION = 0.10
FINAL_ITERATION_STOP_FRACTION = 0.10
FINAL_ITERATION_CALIBRATION_SEED = 42
WANDB_LOG_EVERY = 10

WANDB_PROJECT = "machine-learning"
WANDB_ENTITY = "ml-l"
WANDB_SECRET_NAME = "wandb-api-key"

TARGET_COLUMNS = ["END_LONG", "END_LAT"]
TARGET_SPECS = [
    ("END_LONG", "longitude"),
    ("END_LAT", "latitude"),
]
TARGET_TO_PAYLOAD = {
    "END_LONG": "longitude",
    "END_LAT": "latitude",
}
GROUP_SUFFIX_SEPARATOR = "__prefix_"

FIXED_MODEL_PARAMS = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "task_type": "GPU",
    "devices": "0",
    "grow_policy": "SymmetricTree",
    "boosting_type": "Plain",
    "bootstrap_type": "Bayesian",
    "border_count": 254,
    "one_hot_max_size": 10,
    "max_ctr_complexity": 1,
    "model_size_reg": 0.5,
    "random_seed": SEED,
    "thread_count": -1,
    "allow_writing_files": False,
    "verbose": False,
}

BEST_TARGET_MODEL_PARAMS = {
    "END_LONG": {
        "learning_rate": 0.03,
        "depth": 10,
        "l2_leaf_reg": 12.0,
        "random_strength": 1.0,
        "bagging_temperature": 1.0,
        "one_hot_max_size": 2,
    },
    "END_LAT": {
        "learning_rate": 0.03,
        "depth": 10,
        "l2_leaf_reg": 12.0,
        "random_strength": 1.0,
        "bagging_temperature": 1.0,
        "one_hot_max_size": 2,
    },
}
BEST_TARGET_ITERATIONS = {
    "END_LONG": 5_803,
    "END_LAT": 7_771,
}
BEST_MODEL_RUN_SUMMARY = {
    "catboost_version": "1.2.10",
    "final_fit_completed": False,
    "fit_seconds": 473.0043218135834,
    "generalization_gap_km": 0.34491045996359326,
    "local_submission_rank_by_public_mean": 5,
    "local_submission_all_mean_km": 2.477912305377324,
    "local_submission_all_median_km": 1.3858481907926592,
    "local_submission_all_std_km": 3.1463862311648807,
    "local_submission_all_p90_km": 5.5182050644795995,
    "local_submission_public_mean_km": 2.3903723807692985,
    "local_submission_private_mean_km": 2.5654522299853513,
    "local_submission_n_trips": 320,
    "reviewed_on": "2026-07-15",
    "search_completed": True,
    "selected_model_validation_refit_completed": True,
    "selection_status": "reviewed_hybrid_validation_refit_completed",
    "source": "wandb_export_catboost.csv",
    "source_hybrid_rank": 1,
    "source_run_created_at": "2026-07-14T12:02:05.000Z",
    "source_wandb_run": "catboost-search-summary",
    "target_results": {
        "END_LAT": {
            "best_inner_stop_rmse": 0.025721333,
            "best_iteration": 7770,
            "candidate_index": 13,
            "candidate_label": "cat_13_deep10_ctr",
            "candidate_params": {
                "bagging_temperature": 1.0,
                "depth": 10,
                "l2_leaf_reg": 12.0,
                "learning_rate": 0.03,
                "one_hot_max_size": 2,
                "random_strength": 1.0,
            },
            "ceiling_reached": False,
            "completed": True,
            "completion_state": "completed",
            "early_stopping_fit_seconds": 250.9062681,
            "effective_model_params": {
                "allow_writing_files": False,
                "bagging_temperature": 1.0,
                "boosting_type": "Plain",
                "bootstrap_type": "Bayesian",
                "border_count": 254,
                "depth": 10,
                "devices": "0",
                "eval_metric": "RMSE",
                "grow_policy": "SymmetricTree",
                "iterations": 7771,
                "l2_leaf_reg": 12.0,
                "learning_rate": 0.03,
                "loss_function": "RMSE",
                "max_ctr_complexity": 1,
                "model_size_reg": 0.5,
                "one_hot_max_size": 2,
                "random_seed": 42,
                "random_strength": 1.0,
                "task_type": "GPU",
                "thread_count": -1,
                "verbose": False,
            },
            "elapsed_seconds": 514.2906091213226,
            "eligible_for_hybrid": True,
            "fit_seconds": 256.0874173641205,
            "selected_iterations": 7771,
            "target_column": "END_LAT",
            "target_label": "latitude",
            "trained_iterations": 7871,
            "truncated_at_ceiling": False,
            "validation_refit_fit_seconds": 257.79864048957825,
            "validation_rmse": 0.025969541,
        },
        "END_LONG": {
            "best_inner_stop_rmse": 0.03839646,
            "best_iteration": 5802,
            "candidate_index": 13,
            "candidate_label": "cat_13_deep10_ctr",
            "candidate_params": {
                "bagging_temperature": 1.0,
                "depth": 10,
                "l2_leaf_reg": 12.0,
                "learning_rate": 0.03,
                "one_hot_max_size": 2,
                "random_strength": 1.0,
            },
            "ceiling_reached": False,
            "completed": True,
            "completion_state": "completed",
            "early_stopping_fit_seconds": 186.25644207000732,
            "effective_model_params": {
                "allow_writing_files": False,
                "bagging_temperature": 1.0,
                "boosting_type": "Plain",
                "bootstrap_type": "Bayesian",
                "border_count": 254,
                "depth": 10,
                "devices": "0",
                "eval_metric": "RMSE",
                "grow_policy": "SymmetricTree",
                "iterations": 5803,
                "l2_leaf_reg": 12.0,
                "learning_rate": 0.03,
                "loss_function": "RMSE",
                "max_ctr_complexity": 1,
                "model_size_reg": 0.5,
                "one_hot_max_size": 2,
                "random_seed": 42,
                "random_strength": 1.0,
                "task_type": "GPU",
                "thread_count": -1,
                "verbose": False,
            },
            "elapsed_seconds": 380.43461513519287,
            "eligible_for_hybrid": True,
            "fit_seconds": 216.9169044494629,
            "selected_iterations": 5803,
            "target_column": "END_LONG",
            "target_label": "longitude",
            "trained_iterations": 5903,
            "truncated_at_ceiling": False,
            "validation_refit_fit_seconds": 190.17318630218503,
            "validation_rmse": 0.034763191739519014,
        },
    },
    "train_haversine_mean_km": 1.95180803152413,
    "train_haversine_median_km": 1.2770501264420167,
    "train_haversine_p90_km": 4.159919601337357,
    "train_haversine_std_km": 2.390160575072758,
    "tuner_type": "canonical_summary",
    "validation_haversine_mean_km": 2.2967184914877232,
    "validation_haversine_median_km": 1.3835956643949632,
    "validation_haversine_p90_km": 4.804412155375717,
    "validation_haversine_std_km": 3.39556783408044,
    "validation_refit_fit_seconds": 473.0043218135834,
}

SUBMISSION_FILE = f"submission_{MODEL_SLUG}.csv"
BEST_PARAMS_PATH = ARTIFACT_DIR / f"best_params_{MODEL_SLUG}.json"
MODEL_BUNDLE_PATH = ARTIFACT_DIR / f"model_bundle_{MODEL_SLUG}.joblib"

enabled_expensive_stages = sum(bool(value) for value in (
    RUN_HYPERPARAMETER_TUNING,
    RUN_VALIDATION_REFIT,
    RUN_FINAL_FIT,
))
if enabled_expensive_stages > 1:
    raise ValueError(
        "Enable only one of RUN_HYPERPARAMETER_TUNING, "
        "RUN_VALIDATION_REFIT, or RUN_FINAL_FIT for a Kaggle run."
    )


def require_selected_model_configuration():
    """Validate manually reviewed target parameters and iterations."""
    if set(BEST_TARGET_MODEL_PARAMS) != set(TARGET_COLUMNS):
        raise ValueError("BEST_TARGET_MODEL_PARAMS must cover both targets.")
    if set(BEST_TARGET_ITERATIONS) != set(TARGET_COLUMNS):
        raise ValueError("BEST_TARGET_ITERATIONS must cover both targets.")
    for target_column in TARGET_COLUMNS:
        params = BEST_TARGET_MODEL_PARAMS[target_column]
        iterations = BEST_TARGET_ITERATIONS[target_column]
        if not isinstance(params, dict) or not params:
            raise ValueError(
                f"Missing reviewed parameters for {target_column}."
            )
        if (
            not isinstance(iterations, int)
            or isinstance(iterations, bool)
            or iterations <= 0
        ):
            raise ValueError(
                f"Missing positive reviewed iterations for {target_column}."
            )
    if RUN_FINAL_FIT and not BEST_MODEL_RUN_SUMMARY.get(
        "selected_model_validation_refit_completed",
        False,
    ):
        raise ValueError(
            "Run and review the selected-model validation refit before final fit."
        )
    if RUN_FINAL_FIT:
        required_metrics = [
            "train_haversine_mean_km",
            "train_haversine_median_km",
            "train_haversine_std_km",
            "train_haversine_p90_km",
            "validation_haversine_mean_km",
            "validation_haversine_median_km",
            "validation_haversine_std_km",
            "validation_haversine_p90_km",
            "generalization_gap_km",
        ]
        missing_metrics = [
            key
            for key in required_metrics
            if BEST_MODEL_RUN_SUMMARY.get(key) is None
        ]
        if missing_metrics:
            raise ValueError(
                "Final fit requires reviewed validation metrics: "
                + ", ".join(missing_metrics)
            )


if RUN_VALIDATION_REFIT or RUN_FINAL_FIT:
    require_selected_model_configuration()

## Reproducibility


In [ ]:
def seed_everything(seed=SEED):
    """Seed deterministic Python and NumPy behavior."""
    random.seed(seed)
    np.random.seed(seed)


seed_everything()


## GPU Requirement

Expensive CatBoost modes require a visible CatBoost-compatible GPU and never
fall back to CPU.


In [ ]:
def validate_catboost_gpu_runtime():
    """Record CatBoost runtime details and require one GPU for model work."""
    runtime = {
        "catboost_version": str(catboost.__version__),
        "gpu_checked": False,
        "gpu_device_count": None,
        "gpu_training_nondeterministic": True,
    }
    if enabled_expensive_stages == 0:
        return runtime

    try:
        device_count = int(get_gpu_device_count())
    except Exception as exception:
        raise RuntimeError(
            "A Kaggle GPU accelerator is required. Enable GPU acceleration "
            "before running an expensive CatBoost stage."
        ) from exception

    if device_count < 1:
        raise RuntimeError(
            "No CatBoost-compatible GPU is visible; no CPU fallback is enabled."
        )

    runtime["gpu_checked"] = True
    runtime["gpu_device_count"] = device_count
    return runtime


CATBOOST_RUNTIME = validate_catboost_gpu_runtime()
print("CatBoost version:", CATBOOST_RUNTIME["catboost_version"])
if CATBOOST_RUNTIME["gpu_checked"]:
    print("Visible CatBoost GPU devices:", CATBOOST_RUNTIME["gpu_device_count"])


## Utility Constants


In [ ]:
PREPROCESSED_SCHEMA_VERSION = 2
EXPECTED_DATASET_VARIANT = "catboost"
EXPECTED_MISSING_TOKEN = "__MISSING__"
EXPECTED_CATEGORICAL_ENCODING_POLICY = {
    "CALL_TYPE": "native_string",
    "ORIGIN_CALL": "native_string_plus_missing_indicator_and_log_count",
    "ORIGIN_STAND": "native_string",
    "TAXI_ID": "native_string",
}
CATEGORICAL_FEATURE_COLUMNS = [
    "CALL_TYPE",
    "ORIGIN_CALL",
    "ORIGIN_STAND",
    "TAXI_ID",
]
NUMERIC_METADATA_FEATURE_COLUMNS = [
    "ORIGIN_CALL_MISSING",
    "ORIGIN_CALL_LOG_COUNT",
    "WEEK",
    "DAY",
    "QUARTER",
    "WEEK_SIN",
    "WEEK_COS",
    "DAY_SIN",
    "DAY_COS",
    "QUARTER_SIN",
    "QUARTER_COS",
    "PREFIX_LENGTH",
]
GPS_FEATURE_COLUMNS = [
    feature
    for point_index in range(10)
    for feature in (f"LONG_{point_index}", f"LAT_{point_index}")
]
NUMERICAL_FEATURE_COLUMNS = (
    NUMERIC_METADATA_FEATURE_COLUMNS + GPS_FEATURE_COLUMNS
)
EXPECTED_FEATURE_COLUMNS = (
    NUMERICAL_FEATURE_COLUMNS + CATEGORICAL_FEATURE_COLUMNS
)
EXPECTED_FEATURE_COUNT = 36


## Cached Preprocessed Dataset Helpers


In [ ]:
def require_existing_path(path, description):
    """Return an existing path or fail with an actionable Kaggle message."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}. Attach the schema-v2 CatBoost "
            "Kaggle Dataset or set PREPROCESSED_DATA_DIR."
        )
    return path


def load_preprocessed_metadata(
    base_dir=PREPROCESSED_DATA_DIR,
    require_final=RUN_FINAL_FIT,
):
    """Load and strictly validate mode-appropriate CatBoost metadata."""
    metadata_path = require_existing_path(
        base_dir / "metadata.json",
        "preprocessed metadata",
    )
    with open(metadata_path, "r", encoding="utf-8") as file:
        metadata = json.load(file)

    if metadata.get("schema_version") != PREPROCESSED_SCHEMA_VERSION:
        raise ValueError(
            "The attached preprocessing schema is not CatBoost schema v2."
        )
    if metadata.get("dataset_variant") != EXPECTED_DATASET_VARIANT:
        raise ValueError("The attached dataset_variant is not 'catboost'.")

    constants = metadata.get("constants", {})
    if (
        constants.get("categorical_encoding_policy")
        != EXPECTED_CATEGORICAL_ENCODING_POLICY
    ):
        raise ValueError(
            "The attached native categorical encoding policy does not match."
        )
    if constants.get("categorical_missing_token") != EXPECTED_MISSING_TOKEN:
        raise ValueError("The attached categorical missing token does not match.")
    if (
        constants.get("categorical_feature_columns")
        != CATEGORICAL_FEATURE_COLUMNS
    ):
        raise ValueError("The attached categorical feature names do not match.")
    if constants.get("numerical_feature_columns") != NUMERICAL_FEATURE_COLUMNS:
        raise ValueError("The attached numerical feature names do not match.")

    validation_metadata = metadata.get("validation")
    if not isinstance(validation_metadata, dict):
        raise ValueError("Missing validation preprocessing metadata.")
    if validation_metadata.get("feature_columns") != EXPECTED_FEATURE_COLUMNS:
        raise ValueError(
            "The validation metadata feature contract does not match."
        )

    final_metadata = metadata.get("final")
    if final_metadata is None:
        if require_final:
            raise ValueError(
                "Final-fit mode requires final preprocessing metadata."
            )
    elif not isinstance(final_metadata, dict):
        raise ValueError("Invalid final preprocessing metadata.")
    elif final_metadata.get("feature_columns") != EXPECTED_FEATURE_COLUMNS:
        raise ValueError("The final metadata feature contract does not match.")
    return metadata

def load_cached_frame(relative_path):
    """Load one cached Parquet frame."""
    frame_path = require_existing_path(
        PREPROCESSED_DATA_DIR / relative_path,
        f"cached frame {relative_path}",
    )
    return pd.read_parquet(frame_path)


def assert_index_aligned(features, targets, split_name):
    """Require exact feature/target row alignment."""
    if not features.index.equals(targets.index):
        raise ValueError(
            f"Cached {split_name} features and targets are misaligned."
        )


def assert_matching_features(left, right, left_name, right_name):
    """Require identical ordered feature columns."""
    if not left.columns.equals(right.columns):
        raise ValueError(
            f"Feature columns differ between {left_name} and {right_name}."
        )


def assert_target_columns(targets, split_name):
    """Require targets in cached [LONG, LAT] order."""
    if list(targets.columns) != TARGET_COLUMNS:
        raise ValueError(
            f"Cached {split_name} targets are {list(targets.columns)!r}; "
            f"expected {TARGET_COLUMNS!r}."
        )


def validate_catboost_feature_frame(features, split_name):
    """Validate exact CatBoost feature order, dtypes, and missing policy."""
    if list(features.columns) != EXPECTED_FEATURE_COLUMNS:
        raise ValueError(
            f"Cached {split_name} columns do not match the 36-feature contract."
        )
    if features.shape[1] != EXPECTED_FEATURE_COUNT:
        raise ValueError(
            f"Cached {split_name} has {features.shape[1]} features; "
            f"expected {EXPECTED_FEATURE_COUNT}."
        )
    for column in CATEGORICAL_FEATURE_COLUMNS:
        series = features[column]
        if series.isna().any():
            raise ValueError(
                f"Categorical column {column} contains nulls in {split_name}."
            )
        if not series.map(lambda value: isinstance(value, str)).all():
            raise TypeError(
                f"Categorical column {column} is not string-only in {split_name}."
            )
    for column in NUMERICAL_FEATURE_COLUMNS:
        series = features[column]
        if not pd.api.types.is_numeric_dtype(series.dtype):
            raise TypeError(
                f"Numerical column {column} has dtype {series.dtype} "
                f"in {split_name}."
            )
        if not np.isfinite(
            series.to_numpy(dtype=np.float64, copy=False)
        ).all():
            raise ValueError(
                f"Numerical column {column} is non-finite in {split_name}."
            )


def build_feature_pool(features):
    """Create one feature-only Pool with exact categorical names."""
    return Pool(
        data=features,
        cat_features=CATEGORICAL_FEATURE_COLUMNS,
        feature_names=list(features.columns),
        thread_count=-1,
    )


def build_labeled_pool(features, targets, target_column):
    """Create one target-specific labeled Pool."""
    return Pool(
        data=features,
        label=targets[target_column].to_numpy(dtype=np.float32),
        cat_features=CATEGORICAL_FEATURE_COLUMNS,
        feature_names=list(features.columns),
        thread_count=-1,
    )


def extract_trip_group(index_value, separator=GROUP_SUFFIX_SEPARATOR):
    """Recover the original trip identifier from one cached prefix index."""
    value = str(index_value)
    if separator not in value:
        raise ValueError(
            f"Cached index {value!r} does not contain separator {separator!r}."
        )
    return value.rsplit(separator, 1)[0]


def make_group_disjoint_split(index, stop_fraction, seed):
    """Return deterministic train/stop positions with disjoint trips."""
    if not 0.0 < float(stop_fraction) < 1.0:
        raise ValueError("stop_fraction must be strictly between zero and one.")
    groups = np.asarray(
        [extract_trip_group(value) for value in index],
        dtype=object,
    )
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=float(stop_fraction),
        random_state=int(seed),
    )
    train_positions, stop_positions = next(
        splitter.split(np.zeros(len(index), dtype=np.uint8), groups=groups)
    )
    if set(groups[train_positions]).intersection(groups[stop_positions]):
        raise AssertionError("Original trip groups overlap across partitions.")
    return train_positions, stop_positions, groups


def load_cached_validation_data():
    """Load and validate cached train/validation matrices."""
    X_train = load_cached_frame("validation/X_train.parquet")
    y_train = load_cached_frame("validation/y_train.parquet")
    X_val = load_cached_frame("validation/X_val.parquet")
    y_val = load_cached_frame("validation/y_val.parquet")
    assert_index_aligned(X_train, y_train, "validation train")
    assert_index_aligned(X_val, y_val, "validation holdout")
    assert_matching_features(X_train, X_val, "X_train", "X_val")
    assert_target_columns(y_train, "validation train")
    assert_target_columns(y_val, "validation holdout")
    validate_catboost_feature_frame(X_train, "validation/X_train")
    validate_catboost_feature_frame(X_val, "validation/X_val")
    print("X_train shape:", X_train.shape)
    print("y_train shape:", y_train.shape)
    print("X_val shape:", X_val.shape)
    print("y_val shape:", y_val.shape)
    return X_train, y_train, X_val, y_val


def load_cached_final_data():
    """Load and validate cached full-train and competition-test matrices."""
    final_dir = require_existing_path(
        PREPROCESSED_DATA_DIR / "final",
        "final data directory",
    )
    scaler_path = require_existing_path(
        final_dir / "gps_scaler.joblib",
        "final GPS scaler",
    )
    encoder_path = require_existing_path(
        final_dir / "categorical_encoder.joblib",
        "final categorical encoder",
    )

    X_final_train = load_cached_frame("final/X_train.parquet")
    y_final_train = load_cached_frame("final/y_train.parquet")
    X_final_test = load_cached_frame("final/X_test.parquet")
    final_gps_scaler = joblib.load(scaler_path)
    final_categorical_encoder = joblib.load(encoder_path)

    assert_index_aligned(X_final_train, y_final_train, "final train")
    assert_matching_features(
        X_final_train,
        X_final_test,
        "X_final_train",
        "X_final_test",
    )
    assert_target_columns(y_final_train, "final train")
    validate_catboost_feature_frame(X_final_train, "final/X_train")
    validate_catboost_feature_frame(X_final_test, "final/X_test")
    print("X_final_train shape:", X_final_train.shape)
    print("y_final_train shape:", y_final_train.shape)
    print("X_final_test shape:", X_final_test.shape)
    return (
        X_final_train,
        y_final_train,
        X_final_test,
        final_gps_scaler,
        final_categorical_encoder,
    )


PREPROCESSED_METADATA = load_preprocessed_metadata()


## Evaluation And Submission Helpers


In [ ]:
def make_json_safe(value):
    """Convert notebook values to strict JSON-compatible objects."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, (np.integer, np.floating)):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    if isinstance(value, np.ndarray):
        return make_json_safe(value.tolist())
    if isinstance(value, pd.Series):
        return make_json_safe(value.to_dict())
    if isinstance(value, pd.DataFrame):
        return make_json_safe(value.to_dict(orient="records"))
    if isinstance(value, dict):
        return {str(key): make_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]
    return value


def long_lat_to_lat_long(values):
    """Convert a two-column [LONG, LAT] array to [LAT, LONG]."""
    values = np.asarray(values, dtype=float)
    if values.ndim != 2 or values.shape[1] != 2:
        raise ValueError(f"Expected [LONG, LAT] values; got shape {values.shape}.")
    return np.column_stack([values[:, 1], values[:, 0]])


def combine_target_predictions(predictions_by_target):
    """Combine independent target predictions in [LONG, LAT] order."""
    missing = [target for target in TARGET_COLUMNS if target not in predictions_by_target]
    if missing:
        raise ValueError("Incomplete target predictions: " + ", ".join(missing))
    columns = [
        np.asarray(predictions_by_target[target], dtype=float).reshape(-1)
        for target in TARGET_COLUMNS
    ]
    if len({len(column) for column in columns}) != 1:
        raise ValueError("Longitude and latitude predictions have different lengths.")
    combined = np.column_stack(columns)
    if not np.isfinite(combined).all():
        raise ValueError("Combined predictions contain non-finite values.")
    return combined


def haversine_distances_km(y_true_long_lat, y_pred_long_lat):
    """Return row-wise Haversine errors for [LONG, LAT] arrays."""
    truth = long_lat_to_lat_long(y_true_long_lat)
    prediction = long_lat_to_lat_long(y_pred_long_lat)
    pred_lat = np.radians(prediction[:, 0])
    pred_long = np.radians(prediction[:, 1])
    true_lat = np.radians(truth[:, 0])
    true_long = np.radians(truth[:, 1])
    dlat = true_lat - pred_lat
    dlon = true_long - pred_long
    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(pred_lat) * np.cos(true_lat) * np.sin(dlon / 2.0) ** 2
    )
    a = np.clip(a, 0.0, 1.0)
    return 2.0 * 6371.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))


def evaluate_long_lat_predictions(y_true_long_lat, y_pred_long_lat, prefix):
    """Calculate the standard Haversine metric set."""
    distances = haversine_distances_km(y_true_long_lat, y_pred_long_lat)
    return {
        f"{prefix}/haversine_mean_km": float(np.mean(distances)),
        f"{prefix}/haversine_median_km": float(np.median(distances)),
        f"{prefix}/haversine_std_km": float(np.std(distances)),
        f"{prefix}/haversine_p90_km": float(np.percentile(distances, 90)),
    }, distances


def rmse(y_true, y_pred):
    """Return scalar root mean squared error."""
    truth = np.asarray(y_true, dtype=float).reshape(-1)
    prediction = np.asarray(y_pred, dtype=float).reshape(-1)
    return float(np.sqrt(np.mean((prediction - truth) ** 2)))


def write_submission(trip_ids, destinations, output_path):
    """Write one competition submission from [LAT, LONG] destinations."""
    destinations = np.asarray(destinations, dtype=float)
    expected_shape = (len(trip_ids), 2)
    if destinations.shape != expected_shape:
        raise ValueError(
            f"Expected destination shape {expected_shape}, got {destinations.shape}."
        )
    submission = pd.DataFrame(
        {
            "LATITUDE": destinations[:, 0],
            "LONGITUDE": destinations[:, 1],
        },
        index=trip_ids,
    )
    submission.to_csv(output_path, index_label="TRIP_ID")
    return submission


def save_json_artifact(payload, output_path):
    """Write one JSON artifact."""
    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(
            make_json_safe(payload),
            file,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )


def reviewed_validation_metrics():
    """Return the reviewed metrics currently available in the summary."""
    summary = BEST_MODEL_RUN_SUMMARY
    summary_to_metric = {
        "train_haversine_mean_km": "train/haversine_mean_km",
        "train_haversine_median_km": "train/haversine_median_km",
        "train_haversine_std_km": "train/haversine_std_km",
        "train_haversine_p90_km": "train/haversine_p90_km",
        "validation_haversine_mean_km": "val/haversine_mean_km",
        "validation_haversine_median_km": "val/haversine_median_km",
        "validation_haversine_std_km": "val/haversine_std_km",
        "validation_haversine_p90_km": "val/haversine_p90_km",
        "generalization_gap_km": "generalization/haversine_gap_km",
    }
    metrics = {
        metric_key: summary[summary_key]
        for summary_key, metric_key in summary_to_metric.items()
        if summary.get(summary_key) is not None
    }
    metrics["validation_metrics_source"] = (
        "reviewed_best_model_run_summary"
    )
    return metrics


def save_best_params_artifact(
    validation_metrics,
    target_results,
    execution_status,
    final_target_iterations=None,
    final_iteration_calibration=None,
):
    """Persist reviewed parameters, metrics, iterations, and status."""
    final_iterations = (
        None
        if final_target_iterations is None
        else {
            target: int(iterations)
            for target, iterations in final_target_iterations.items()
        }
    )
    run_summary = deepcopy(BEST_MODEL_RUN_SUMMARY)
    if execution_status.get("validation_refit_completed", False):
        aggregate_fit_seconds = float(validation_metrics["fit_seconds"])
        run_summary.update({
            "selection_status": "reviewed_hybrid_validation_refit_completed",
            "selected_model_validation_refit_completed": True,
            "train_haversine_mean_km": validation_metrics[
                "train/haversine_mean_km"
            ],
            "train_haversine_median_km": validation_metrics[
                "train/haversine_median_km"
            ],
            "train_haversine_std_km": validation_metrics[
                "train/haversine_std_km"
            ],
            "train_haversine_p90_km": validation_metrics[
                "train/haversine_p90_km"
            ],
            "validation_haversine_mean_km": validation_metrics[
                "val/haversine_mean_km"
            ],
            "validation_haversine_median_km": validation_metrics[
                "val/haversine_median_km"
            ],
            "validation_haversine_std_km": validation_metrics[
                "val/haversine_std_km"
            ],
            "validation_haversine_p90_km": validation_metrics[
                "val/haversine_p90_km"
            ],
            "generalization_gap_km": validation_metrics[
                "generalization/haversine_gap_km"
            ],
            "fit_seconds": aggregate_fit_seconds,
            "validation_refit_fit_seconds": aggregate_fit_seconds,
            "catboost_version": CATBOOST_RUNTIME["catboost_version"],
        })
        reviewed_target_results = run_summary.setdefault(
            "target_results",
            {},
        )
        for target_column in TARGET_COLUMNS:
            reviewed_target = reviewed_target_results.get(
                target_column,
                {},
            )
            reviewed_target_results[target_column] = {
                **reviewed_target,
                **deepcopy(target_results[target_column]),
            }

    payload = {
        "model_name": MODEL_NAME,
        "model_slug": MODEL_SLUG,
        "params_source": "reviewed_target_specific_hybrid",
        "saved_target_model_params": BEST_TARGET_MODEL_PARAMS,
        "saved_target_iterations": BEST_TARGET_ITERATIONS,
        "reviewed_target_iterations": BEST_TARGET_ITERATIONS,
        "final_target_iterations": final_iterations,
        "final_iteration_calibration": final_iteration_calibration,
        "best_model_run_summary": run_summary,
        "fixed_model_params": FIXED_MODEL_PARAMS,
        "categorical_feature_columns": CATEGORICAL_FEATURE_COLUMNS,
        "numerical_feature_columns": NUMERICAL_FEATURE_COLUMNS,
        "target_results": target_results,
        "validation_metrics": validation_metrics,
        "execution": execution_status,
        "catboost_runtime": CATBOOST_RUNTIME,
        "preprocessed_dataset": {
            "path": str(PREPROCESSED_DATA_DIR),
            "schema_version": PREPROCESSED_METADATA.get("schema_version"),
            "dataset_variant": PREPROCESSED_METADATA.get("dataset_variant"),
            "config": PREPROCESSED_METADATA.get("config", {}),
            "validation": PREPROCESSED_METADATA.get("validation", {}),
            "final": PREPROCESSED_METADATA.get("final", {}),
        },
    }
    save_json_artifact(payload, BEST_PARAMS_PATH)
    return payload


## Define CatBoost Model


In [ ]:
def effective_model_params(candidate_params, iterations):
    """Return the exact merged CatBoostRegressor configuration."""
    return {
        **FIXED_MODEL_PARAMS,
        **candidate_params,
        "iterations": int(iterations),
    }


def build_model(candidate_params, iterations):
    """Build one GPU CatBoost regressor."""
    return CatBoostRegressor(
        **effective_model_params(candidate_params, iterations)
    )


def fit_pair_with_fixed_iterations(
    features,
    targets,
    model_params_by_target,
    target_iterations,
):
    """Fit a complete target pair with reviewed fixed iterations."""
    models = {}
    target_results = {}
    failed_target = None
    try:
        for target_column, target_label in TARGET_SPECS:
            failed_target = target_column
            selected_iterations = int(target_iterations[target_column])
            model_params = model_params_by_target[target_column]
            training_pool = build_labeled_pool(
                features,
                targets,
                target_column,
            )
            model = build_model(model_params, selected_iterations)
            fit_started = time.time()
            model.fit(training_pool, verbose=False)
            fit_seconds = float(time.time() - fit_started)
            models[target_column] = model
            target_results[target_column] = {
                "target_column": target_column,
                "target_label": target_label,
                "completed": True,
                "candidate_params": model_params.copy(),
                "selected_iterations": selected_iterations,
                "fit_seconds": fit_seconds,
                "effective_model_params": effective_model_params(
                    model_params,
                    selected_iterations,
                ),
            }
            del training_pool
            gc.collect()
        return models, target_results
    except Exception as exception:
        models.clear()
        gc.collect()
        raise RuntimeError(
            "The fixed-iteration estimator pair is incomplete; "
            f"target {failed_target!r} did not finish successfully."
        ) from exception


def serialize_catboost_model_to_cbm(model):
    """Return one model as native CBM bytes."""
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            suffix=".cbm",
            dir=ARTIFACT_DIR,
            delete=False,
        ) as temporary_file:
            temporary_path = Path(temporary_file.name)
        model.save_model(str(temporary_path), format="cbm")
        return temporary_path.read_bytes()
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


def restore_catboost_model_from_cbm(payload):
    """Restore one CatBoostRegressor from native CBM bytes."""
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            suffix=".cbm",
            dir=ARTIFACT_DIR,
            delete=False,
        ) as temporary_file:
            temporary_path = Path(temporary_file.name)
            temporary_file.write(payload)
        model = CatBoostRegressor()
        model.load_model(str(temporary_path), format="cbm")
        return model
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


def verify_cbm_payload_predictions(payload, prediction_pool, expected):
    """Require restored-CBM predictions to match the live model."""
    restored_model = restore_catboost_model_from_cbm(payload)
    try:
        restored_prediction = np.asarray(
            restored_model.predict(prediction_pool),
            dtype=float,
        )
        if not np.allclose(
            restored_prediction,
            np.asarray(expected, dtype=float),
            rtol=1e-7,
            atol=1e-7,
        ):
            raise AssertionError(
                "Restored CBM predictions differ from live-model predictions."
            )
    finally:
        del restored_model
        gc.collect()


def build_final_model_bundle(
    model_payloads,
    target_results,
    gps_scaler,
    categorical_encoder,
    feature_columns,
    validation_metrics,
    final_target_iterations,
    final_iteration_calibration,
    execution_status,
):
    """Build the stable target-specific CatBoost model bundle."""
    final_iterations = {
        target: int(iterations)
        for target, iterations in final_target_iterations.items()
    }
    return {
        "model_name": MODEL_NAME,
        "model_slug": MODEL_SLUG,
        "model_serialized": True,
        "model_format": "cbm_bytes",
        "model_payloads": model_payloads,
        "target_to_payload": TARGET_TO_PAYLOAD.copy(),
        "target_columns": TARGET_COLUMNS.copy(),
        "target_specs": TARGET_SPECS.copy(),
        "categorical_feature_columns": CATEGORICAL_FEATURE_COLUMNS.copy(),
        "numerical_feature_columns": NUMERICAL_FEATURE_COLUMNS.copy(),
        "model_params_by_target": deepcopy(BEST_TARGET_MODEL_PARAMS),
        "fixed_model_params": FIXED_MODEL_PARAMS.copy(),
        "target_iterations": final_iterations.copy(),
        "reviewed_target_iterations": BEST_TARGET_ITERATIONS.copy(),
        "final_target_iterations": final_iterations.copy(),
        "final_iteration_calibration": deepcopy(
            final_iteration_calibration
        ),
        "target_results": target_results,
        "gps_scaler": gps_scaler,
        "categorical_encoder": categorical_encoder,
        "feature_columns": list(feature_columns),
        "validation_metrics": validation_metrics.copy(),
        "selected_best_run_summary": deepcopy(BEST_MODEL_RUN_SUMMARY),
        "execution": deepcopy(execution_status),
        "final_refit": True,
        "prediction_format": "[LONG, LAT]",
        "preprocessed_data_dir": str(PREPROCESSED_DATA_DIR),
        "preprocessed_schema_version": PREPROCESSED_METADATA.get(
            "schema_version"
        ),
        "preprocessed_metadata": PREPROCESSED_METADATA,
        "catboost_runtime": CATBOOST_RUNTIME,
    }


## Hyperparameter Tuning

Use one grouped inner split only to select target-specific iteration counts.
Refit every candidate on all validation-training rows, then select
longitude/latitude sources by Haversine distance on `validation/X_val`.
Results remain unreviewed until manually promoted into the constants above.


### Search Catalogue

Keep the complete 18-candidate catalogue inline so the Kaggle search is
self-contained and reproducible. The catalogue is evaluated only when
`RUN_HYPERPARAMETER_TUNING=True`.


In [ ]:
BALANCED_PARAMS = {
    "learning_rate": 0.05,
    "depth": 8,
    "l2_leaf_reg": 5.0,
    "random_strength": 1.0,
    "bagging_temperature": 1.0,
}


def candidate(candidate_index, candidate_label, **overrides):
    """Return one finite CatBoost catalogue entry."""
    return {
        "candidate_index": candidate_index,
        "candidate_label": candidate_label,
        "params": {**BALANCED_PARAMS, **overrides},
    }


TUNING_CANDIDATES = [
    candidate(1, "cat_01_balanced_baseline"),
    candidate(
        2,
        "cat_02_shallow",
        depth=6,
        l2_leaf_reg=3.0,
    ),
    candidate(3, "cat_03_depth7", depth=7),
    candidate(
        4,
        "cat_04_deep9_regularized",
        depth=9,
        l2_leaf_reg=8.0,
    ),
    candidate(
        5,
        "cat_05_deep10_slow",
        learning_rate=0.03,
        depth=10,
        l2_leaf_reg=12.0,
    ),
    candidate(6, "cat_06_slow_learning", learning_rate=0.03),
    candidate(7, "cat_07_fast_learning", learning_rate=0.08),
    candidate(
        8,
        "cat_08_low_regularization",
        l2_leaf_reg=2.0,
        random_strength=0.5,
        bagging_temperature=0.5,
    ),
    candidate(
        9,
        "cat_09_strong_regularization",
        l2_leaf_reg=15.0,
        random_strength=2.0,
        bagging_temperature=2.0,
    ),
    candidate(
        10,
        "cat_10_low_randomness",
        random_strength=0.25,
        bagging_temperature=0.0,
    ),
    candidate(
        11,
        "cat_11_call_type_ctr",
        one_hot_max_size=2,
    ),
    candidate(
        12,
        "cat_12_ctr_pair_interactions",
        learning_rate=0.04,
        l2_leaf_reg=8.0,
        max_ctr_complexity=2,
    ),
    candidate(
        13,
        "cat_13_deep10_ctr",
        learning_rate=0.03,
        depth=10,
        l2_leaf_reg=12.0,
        one_hot_max_size=2,
    ),
    candidate(
        14,
        "cat_14_fast_ctr",
        learning_rate=0.08,
        one_hot_max_size=2,
    ),
    candidate(
        15,
        "cat_15_low_noise_low_reg",
        l2_leaf_reg=2.0,
        random_strength=0.25,
        bagging_temperature=0.0,
    ),
    candidate(
        16,
        "cat_16_low_noise_ctr",
        random_strength=0.25,
        bagging_temperature=0.0,
        one_hot_max_size=2,
    ),
    candidate(
        17,
        "cat_17_deep10_mid_lr",
        learning_rate=0.04,
        depth=10,
        l2_leaf_reg=12.0,
    ),
    candidate(
        18,
        "cat_18_deep10_mid_lr_ctr",
        learning_rate=0.04,
        depth=10,
        l2_leaf_reg=12.0,
        one_hot_max_size=2,
    ),
]

def canonical_parameter_signature(params):
    """Return a numeric-type-insensitive signature for a flat parameter map."""
    normalized = {}
    for key, value in params.items():
        if isinstance(value, bool) or value is None or isinstance(value, str):
            normalized[key] = value
        elif isinstance(value, (int, float, np.integer, np.floating)):
            normalized[key] = float(value)
        else:
            normalized[key] = value
    return json.dumps(normalized, sort_keys=True, separators=(",", ":"))


TUNING_UNIQUE_CANDIDATE_COUNT = len(TUNING_CANDIDATES)
TUNING_CANDIDATE_INDICES = [
    item["candidate_index"] for item in TUNING_CANDIDATES
]
if RUN_HYPERPARAMETER_TUNING:
    if TUNING_UNIQUE_CANDIDATE_COUNT != 18:
        raise RuntimeError("The CatBoost catalogue must contain 18 candidates.")
    if TUNING_CANDIDATE_INDICES != list(range(1, 19)):
        raise RuntimeError("CatBoost candidate indices must be exactly 1–18.")
    if len({item["candidate_label"] for item in TUNING_CANDIDATES}) != 18:
        raise RuntimeError("CatBoost candidate labels must be unique.")
    if len({canonical_parameter_signature(item["params"]) for item in TUNING_CANDIDATES}) != 18:
        raise RuntimeError("CatBoost candidate parameter configurations must be numerically unique.")


### Tuning Logging Helpers

Validate W&B once before loading the large search matrices. Logging failures
never control model training. CatBoost histories are uploaded after a completed
fit, and history upload stops after the first failed W&B log call.


In [ ]:
WANDB_ENABLED = False
_WANDB_MODULE = None
_WANDB_MODE = "disabled"
_WANDB_WARNING_STAGES = set()


def get_wandb_key():
    """Return the W&B API key from Kaggle secrets or the environment."""
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(WANDB_SECRET_NAME)
    except Exception:
        return os.environ.get("WANDB_API_KEY")


def wandb_warning(stage, exception):
    """Emit one sanitized warning per W&B stage."""
    if stage not in _WANDB_WARNING_STAGES:
        print(
            f"W&B {stage} failed ({type(exception).__name__}). "
            "Model tuning will continue."
        )
        _WANDB_WARNING_STAGES.add(stage)


def initialize_wandb():
    """Validate W&B credentials once before optional tuning starts."""
    wandb_key = get_wandb_key()
    if not wandb_key:
        print("No W&B key found. Tuning will run without W&B logging.")
        return None, "disabled", False
    try:
        import wandb

        login_succeeded = wandb.login(key=wandb_key)
        if login_succeeded is False:
            raise RuntimeError("W&B login was rejected.")
        return wandb, "online", True
    except Exception as exception:
        wandb_warning("setup", exception)
        return None, "disabled", False


if RUN_HYPERPARAMETER_TUNING:
    _WANDB_MODULE, _WANDB_MODE, WANDB_ENABLED = initialize_wandb()


def target_candidate_trial_index(candidate_item, target_column):
    """Return the unique 1-based W&B trial index for one target run."""
    if target_column not in TARGET_COLUMNS:
        raise ValueError(f"Unknown target column: {target_column!r}.")
    candidate_index = int(candidate_item["candidate_index"])
    if candidate_index not in TUNING_CANDIDATE_INDICES:
        raise ValueError(f"Invalid candidate index: {candidate_index}.")
    candidate_position = TUNING_CANDIDATE_INDICES.index(candidate_index) + 1
    target_offset = (
        TARGET_COLUMNS.index(target_column)
        * TUNING_UNIQUE_CANDIDATE_COUNT
    )
    return target_offset + candidate_position


def initialize_target_candidate_wandb_run(
    candidate_item,
    target_column,
    target_label,
):
    """Start one optional W&B run for a target-candidate evaluation."""
    if not WANDB_ENABLED:
        return None
    run = None
    try:
        run = _WANDB_MODULE.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            name=(
                f"catboost-search-{candidate_item['candidate_label']}-"
                f"{target_label}"
            ),
            job_type="catboost_target_candidate",
            mode=_WANDB_MODE,
            config={
                "model_slug": MODEL_SLUG,
                "trial_index": target_candidate_trial_index(
                    candidate_item,
                    target_column,
                ),
                "trial_params": candidate_item["params"],
                "tuner_type": "canonical_search",
                "candidate_index": candidate_item["candidate_index"],
                "candidate_label": candidate_item["candidate_label"],
                "candidate_params": candidate_item["params"],
                "target_column": target_column,
                "target_label": target_label,
                "fixed_model_params": FIXED_MODEL_PARAMS,
                "max_iterations": MAX_ITERATIONS,
                "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
                "search_stop_fraction": SEARCH_STOP_FRACTION,
                "seed": SEED,
            },
            reinit=True,
        )
        if run is None:
            raise RuntimeError("wandb.init returned no target-candidate run.")
        iteration_metric = f"{target_label}/iteration"
        run.define_metric(iteration_metric)
        run.define_metric(
            f"{target_label}/inner_train_rmse",
            step_metric=iteration_metric,
            summary="min",
        )
        run.define_metric(
            f"{target_label}/inner_stop_rmse",
            step_metric=iteration_metric,
            summary="min",
        )
        return run
    except Exception as exception:
        wandb_warning("target-candidate setup", exception)
        if run is not None:
            try:
                run.finish()
            except Exception as finish_exception:
                wandb_warning(
                    "target-candidate setup finalization",
                    finish_exception,
                )
        return None


def safe_wandb_summary(run, payload):
    """Write optional summary values without affecting search."""
    if run is None:
        return
    try:
        for key, value in make_json_safe(payload).items():
            run.summary[key] = value
    except Exception as exception:
        wandb_warning("summary", exception)


def safe_wandb_log(run, payload):
    """Log optional values and report whether the write succeeded."""
    if run is None:
        return False
    try:
        run.log(payload)
        return True
    except Exception as exception:
        wandb_warning("logging", exception)
        return False

def safe_wandb_finish(run):
    """Finish an optional run without masking model results."""
    if run is None:
        return
    try:
        run.finish()
    except Exception as exception:
        wandb_warning("finalization", exception)


def metric_history(evaluation_history, dataset_name):
    """Return an RMSE history regardless of CatBoost metric key casing."""
    metrics = evaluation_history.get(dataset_name, {})
    for metric_name, values in metrics.items():
        if metric_name.lower() == "rmse":
            return [float(value) for value in values]
    return []


def log_catboost_history_to_wandb(
    run,
    target_label,
    evaluation_history,
    best_iteration,
):
    """Upload bounded post-fit history, stopping after the first failure."""
    if run is None:
        return False
    train_history = metric_history(evaluation_history, "learn")
    stop_history = metric_history(evaluation_history, "validation")
    history_length = max(len(train_history), len(stop_history))
    if history_length == 0:
        return True
    selected_steps = set(range(0, history_length, WANDB_LOG_EVERY))
    selected_steps.update({int(best_iteration), history_length - 1})
    for iteration in sorted(
        step for step in selected_steps if 0 <= step < history_length
    ):
        payload = {f"{target_label}/iteration": int(iteration)}
        if iteration < len(train_history):
            payload[f"{target_label}/inner_train_rmse"] = train_history[
                iteration
            ]
        if iteration < len(stop_history):
            payload[f"{target_label}/inner_stop_rmse"] = stop_history[
                iteration
            ]
        if not safe_wandb_log(run, payload):
            return False
    return True

def log_search_summary_to_wandb(target_results, hybrid_results, winner):
    """Log target-candidate and hybrid search tables."""
    if not WANDB_ENABLED:
        return
    run = None
    try:
        run = _WANDB_MODULE.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            name="catboost-search-summary",
            job_type="catboost_search_summary",
            mode=_WANDB_MODE,
            config={
                "model_slug": MODEL_SLUG,
                "seed": SEED,
                "tuner_type": "canonical_summary",
                "candidate_count": TUNING_UNIQUE_CANDIDATE_COUNT,
                "target_candidate_count": len(target_results),
                "hybrid_count": len(hybrid_results),
            },
            reinit=True,
        )
        if run is None:
            raise RuntimeError("wandb.init returned no summary run.")
        run.log({
            "target_candidate_results": _WANDB_MODULE.Table(
                dataframe=target_results
            ),
            "hybrid_results": _WANDB_MODULE.Table(
                dataframe=hybrid_results
            ),
        })
        safe_wandb_summary(run, {
            "winning_longitude_candidate": winner[
                "longitude_candidate_label"
            ],
            "winning_latitude_candidate": winner[
                "latitude_candidate_label"
            ],
            "validation_haversine_mean_km": winner[
                "haversine_mean_km"
            ],
            "validation_haversine_p90_km": winner[
                "haversine_p90_km"
            ],
            "hybrid_count": len(hybrid_results),
            "promotion_status": "requires_manual_review",
        })
    except Exception as exception:
        wandb_warning("search summary", exception)
    finally:
        safe_wandb_finish(run)


### Candidate Evaluation


In [ ]:
def fit_target_with_early_stopping(
    model_params,
    target_column,
    target_label,
    training_pool,
    stop_pool,
    run=None,
):
    """Select one target's iterations using CatBoost's detector."""
    model = build_model(model_params, MAX_ITERATIONS)
    try:
        fit_started = time.time()
        model.fit(
            training_pool,
            eval_set=stop_pool,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            use_best_model=True,
            verbose=False,
        )
        fit_seconds = float(time.time() - fit_started)
        best_iteration = model.get_best_iteration()
        if best_iteration is None or int(best_iteration) < 0:
            raise RuntimeError("CatBoost did not report a best iteration.")
        best_iteration = int(best_iteration)
        selected_iterations = best_iteration + 1
        evaluation_history = model.get_evals_result()
        stop_history = metric_history(evaluation_history, "validation")
        trained_iterations = len(stop_history)
        patience_observed = trained_iterations - selected_iterations
        ceiling_reached = trained_iterations >= MAX_ITERATIONS
        truncated_at_ceiling = bool(
            ceiling_reached
            and patience_observed < EARLY_STOPPING_ROUNDS
        )
        completion_state = (
            "truncated_at_ceiling"
            if truncated_at_ceiling
            else "completed"
        )
        history_logging_succeeded = log_catboost_history_to_wandb(
            run,
            target_label,
            evaluation_history,
            best_iteration,
        )
        best_score = model.get_best_score()
        validation_scores = best_score.get("validation", {})
        best_inner_stop_rmse = next(
            (
                float(value)
                for metric_name, value in validation_scores.items()
                if metric_name.lower() == "rmse"
            ),
            None,
        )
        return {
            "target_column": target_column,
            "target_label": target_label,
            "completed": True,
            "completion_state": completion_state,
            "eligible_for_hybrid": not truncated_at_ceiling,
            "candidate_params": model_params.copy(),
            "best_iteration": best_iteration,
            "selected_iterations": selected_iterations,
            "trained_iterations": trained_iterations,
            "patience_iterations_observed": patience_observed,
            "best_inner_stop_rmse": best_inner_stop_rmse,
            "early_stopping_fit_seconds": fit_seconds,
            "ceiling_reached": ceiling_reached,
            "truncated_at_ceiling": truncated_at_ceiling,
            "wandb_history_logging_succeeded": (
                history_logging_succeeded
            ),
            "effective_model_params": effective_model_params(
                model_params,
                selected_iterations,
            ),
        }
    finally:
        del model
        gc.collect()

def evaluate_target_candidate(
    candidate_item,
    target_column,
    target_label,
    full_training_pool,
    inner_training_pool,
    inner_stop_pool,
    validation_pool,
    validation_truth,
):
    """Early-stop, refit, and evaluate one target-candidate."""
    run = initialize_target_candidate_wandb_run(
        candidate_item,
        target_column,
        target_label,
    )
    evaluation_started = time.time()
    result = {
        "candidate_index": candidate_item["candidate_index"],
        "candidate_label": candidate_item["candidate_label"],
        "target_column": target_column,
        "target_label": target_label,
        "candidate_params": candidate_item["params"].copy(),
        "completed": False,
        "completion_state": "failed",
        "eligible_for_hybrid": False,
        "validation_refit_completed": False,
        "validation_refit_completion_state": "not_run",
        "validation_prediction": None,
    }
    try:
        selection_result = fit_target_with_early_stopping(
            candidate_item["params"],
            target_column,
            target_label,
            inner_training_pool,
            inner_stop_pool,
            run=run,
        )
        result.update(selection_result)
        result.update({
            "validation_refit_fit_seconds": None,
            "validation_rmse": None,
        })
        prediction = None
        if result["eligible_for_hybrid"]:
            refit_model = build_model(
                candidate_item["params"],
                result["selected_iterations"],
            )
            try:
                fit_started = time.time()
                refit_model.fit(full_training_pool, verbose=False)
                result["validation_refit_fit_seconds"] = float(
                    time.time() - fit_started
                )
                prediction = np.asarray(
                    refit_model.predict(validation_pool),
                    dtype=np.float32,
                )
            finally:
                del refit_model
                gc.collect()
            result.update({
                "validation_refit_completed": True,
                "validation_refit_completion_state": "completed",
                "validation_rmse": rmse(validation_truth, prediction),
            })
        else:
            result["validation_refit_completion_state"] = (
                "not_run_truncated"
            )

        result["fit_seconds"] = result.get(
            "validation_refit_fit_seconds"
        )
        result["elapsed_seconds"] = float(
            time.time() - evaluation_started
        )
        summary_payload = {
            "execution_completed": True,
            "completion_state": result["completion_state"],
            "eligible_for_hybrid": result["eligible_for_hybrid"],
            "best_iteration": result.get("best_iteration"),
            "selected_iterations": result.get("selected_iterations"),
            "trained_iterations": result.get("trained_iterations"),
            "best_inner_stop_rmse": result.get(
                "best_inner_stop_rmse"
            ),
            "validation_rmse": result.get("validation_rmse"),
            "early_stopping_fit_seconds": result.get(
                "early_stopping_fit_seconds"
            ),
            "validation_refit_fit_seconds": result.get(
                "validation_refit_fit_seconds"
            ),
            "fit_seconds": result.get("fit_seconds"),
            "elapsed_seconds": result.get("elapsed_seconds"),
            "ceiling_reached": result.get("ceiling_reached"),
            "truncated_at_ceiling": result.get(
                "truncated_at_ceiling"
            ),
        }
        safe_wandb_summary(run, summary_payload)
        if (
            result.get("validation_refit_completed") is True
            and result.get("wandb_history_logging_succeeded") is True
        ):
            safe_wandb_log(run, {
                "validation_rmse": result["validation_rmse"],
                "fit_seconds": result["fit_seconds"],
                "elapsed_seconds": result["elapsed_seconds"],
            })
        return result, prediction
    except Exception as exception:
        result.update({
            "completed": False,
            "completion_state": "failed",
            "eligible_for_hybrid": False,
            "validation_refit_completed": False,
            "validation_refit_completion_state": "failed",
            "fit_seconds": result.get("validation_refit_fit_seconds"),
            "elapsed_seconds": float(time.time() - evaluation_started),
            "failure_type": type(exception).__name__,
            "failure_message": str(exception),
        })
        safe_wandb_summary(run, {
            "execution_completed": False,
            "completion_state": "failed",
            "fit_seconds": result.get("fit_seconds"),
            "elapsed_seconds": result["elapsed_seconds"],
            "failure_type": type(exception).__name__,
        })
        return result, None
    finally:
        safe_wandb_finish(run)
        gc.collect()

def evaluate_target_catalogue(
    target_column,
    target_label,
    X_train,
    y_train,
    y_val,
    train_positions,
    stop_positions,
    validation_pool,
):
    """Evaluate all candidates while reusing target-specific Pools."""
    full_training_pool = build_labeled_pool(
        X_train,
        y_train,
        target_column,
    )
    inner_training_pool = full_training_pool.slice(train_positions)
    inner_stop_pool = full_training_pool.slice(stop_positions)
    validation_truth = y_val[target_column].to_numpy(dtype=np.float32)
    records = {}
    predictions = {}
    try:
        for candidate_position, candidate_item in enumerate(
            TUNING_CANDIDATES,
            start=1,
        ):
            result, prediction = evaluate_target_candidate(
                candidate_item,
                target_column,
                target_label,
                full_training_pool,
                inner_training_pool,
                inner_stop_pool,
                validation_pool,
                validation_truth,
            )
            label = candidate_item["candidate_label"]
            records[label] = result
            if prediction is not None:
                predictions[label] = prediction
            result_text = (
                f"{result['validation_rmse']:.8f} RMSE"
                if result.get("validation_rmse") is not None
                else result["completion_state"]
            )
            print(
                f"[{target_label} {candidate_position:02d}/"
                f"{TUNING_UNIQUE_CANDIDATE_COUNT}] "
                f"{label}: {result_text}"
            )
    finally:
        del full_training_pool, inner_training_pool, inner_stop_pool
        gc.collect()
    return records, predictions


def target_result_to_row(result):
    """Flatten one target-candidate result for tables and W&B."""
    return {
        "candidate_index": result.get("candidate_index"),
        "candidate_label": result.get("candidate_label"),
        "target_column": result.get("target_column"),
        "target_label": result.get("target_label"),
        "completion_state": result.get("completion_state"),
        "eligible_for_hybrid": result.get("eligible_for_hybrid", False),
        "best_iteration": result.get("best_iteration"),
        "selected_iterations": result.get("selected_iterations"),
        "trained_iterations": result.get("trained_iterations"),
        "best_inner_stop_rmse": result.get("best_inner_stop_rmse"),
        "validation_rmse": result.get("validation_rmse"),
        "ceiling_reached": result.get("ceiling_reached"),
        "truncated_at_ceiling": result.get("truncated_at_ceiling"),
        "early_stopping_fit_seconds": result.get(
            "early_stopping_fit_seconds"
        ),
        "validation_refit_fit_seconds": result.get(
            "validation_refit_fit_seconds"
        ),
        "fit_seconds": result.get("fit_seconds"),
        "elapsed_seconds": result.get("elapsed_seconds"),
        "candidate_params": repr(result.get("candidate_params", {})),
        "failure_type": result.get("failure_type"),
    }


### Hybrid Selection


In [ ]:
HYBRID_SORT_COLUMNS = [
    "haversine_mean_km",
    "haversine_p90_km",
    "haversine_median_km",
    "total_selected_iterations",
    "longitude_candidate_label",
    "latitude_candidate_label",
]


def eligible_target_labels(candidate_catalog, records, predictions):
    """Return catalogue-ordered labels with complete X_val predictions."""
    labels = []
    for candidate_item in candidate_catalog:
        label = candidate_item["candidate_label"]
        result = records.get(label, {})
        if (
            result.get("completed") is True
            and result.get("eligible_for_hybrid") is True
            and result.get("completion_state") == "completed"
            and result.get("validation_refit_completed") is True
            and result.get("validation_refit_completion_state")
            == "completed"
            and label in predictions
        ):
            labels.append(label)
    return labels


def enumerate_hybrid_rows(
    longitude_labels,
    latitude_labels,
    predictions_by_target,
    records_by_target,
    truth_long_lat,
):
    """Evaluate every eligible target-specific hybrid on X_val."""
    rows = []
    for longitude_label in longitude_labels:
        longitude_prediction = predictions_by_target["END_LONG"][
            longitude_label
        ]
        longitude_result = records_by_target["END_LONG"][
            longitude_label
        ]
        for latitude_label in latitude_labels:
            latitude_prediction = predictions_by_target["END_LAT"][
                latitude_label
            ]
            latitude_result = records_by_target["END_LAT"][
                latitude_label
            ]
            combined = combine_target_predictions({
                "END_LONG": longitude_prediction,
                "END_LAT": latitude_prediction,
            })
            metrics, _ = evaluate_long_lat_predictions(
                truth_long_lat,
                combined,
                "validation",
            )
            rows.append({
                "longitude_candidate_label": longitude_label,
                "latitude_candidate_label": latitude_label,
                "longitude_selected_iterations": longitude_result[
                    "selected_iterations"
                ],
                "latitude_selected_iterations": latitude_result[
                    "selected_iterations"
                ],
                "total_selected_iterations": int(
                    longitude_result["selected_iterations"]
                    + latitude_result["selected_iterations"]
                ),
                "haversine_mean_km": metrics[
                    "validation/haversine_mean_km"
                ],
                "haversine_median_km": metrics[
                    "validation/haversine_median_km"
                ],
                "haversine_std_km": metrics[
                    "validation/haversine_std_km"
                ],
                "haversine_p90_km": metrics[
                    "validation/haversine_p90_km"
                ],
            })
    return rows


def deterministically_rank_hybrids(hybrid_rows):
    """Sort hybrid evidence using the complete deterministic contract."""
    result = pd.DataFrame(hybrid_rows)
    if result.empty:
        raise RuntimeError("No eligible CatBoost hybrid was generated.")
    result = result.sort_values(
        HYBRID_SORT_COLUMNS,
        ascending=True,
        kind="mergesort",
    ).reset_index(drop=True)
    result.insert(0, "hybrid_rank", np.arange(1, len(result) + 1))
    return result


### Run Hyperparameter Tuning


In [ ]:
def run_hyperparameter_tuning():
    """Evaluate target candidates, then rank every eligible hybrid."""
    if not RUN_HYPERPARAMETER_TUNING:
        print("Hyperparameter tuning disabled. Using reviewed parameters.")
        return pd.DataFrame(), pd.DataFrame(), None

    X_train, y_train, X_val, y_val = load_cached_validation_data()
    train_positions, stop_positions, trip_groups = make_group_disjoint_split(
        X_train.index,
        stop_fraction=SEARCH_STOP_FRACTION,
        seed=SEED,
    )
    train_group_count = len(set(trip_groups[train_positions]))
    stop_group_count = len(set(trip_groups[stop_positions]))
    if set(trip_groups[train_positions]).intersection(
        trip_groups[stop_positions]
    ):
        raise AssertionError("Inner CatBoost trip groups overlap.")
    del trip_groups
    gc.collect()

    display(pd.Series({
        "validation_train_rows": len(X_train),
        "inner_train_rows": len(train_positions),
        "inner_stop_rows": len(stop_positions),
        "validation_holdout_rows": len(X_val),
        "inner_train_trip_groups": train_group_count,
        "inner_stop_trip_groups": stop_group_count,
        "overlapping_trip_groups": 0,
    }))

    validation_pool = build_feature_pool(X_val)
    records_by_target = {}
    predictions_by_target = {}
    try:
        for target_column, target_label in TARGET_SPECS:
            records, predictions = evaluate_target_catalogue(
                target_column,
                target_label,
                X_train,
                y_train,
                y_val,
                train_positions,
                stop_positions,
                validation_pool,
            )
            records_by_target[target_column] = records
            predictions_by_target[target_column] = predictions
    finally:
        del validation_pool
        gc.collect()

    longitude_labels = eligible_target_labels(
        TUNING_CANDIDATES,
        records_by_target["END_LONG"],
        predictions_by_target["END_LONG"],
    )
    latitude_labels = eligible_target_labels(
        TUNING_CANDIDATES,
        records_by_target["END_LAT"],
        predictions_by_target["END_LAT"],
    )
    if not longitude_labels:
        raise RuntimeError("No eligible longitude candidate completed.")
    if not latitude_labels:
        raise RuntimeError("No eligible latitude candidate completed.")

    hybrid_rows = enumerate_hybrid_rows(
        longitude_labels,
        latitude_labels,
        predictions_by_target,
        records_by_target,
        y_val[TARGET_COLUMNS].to_numpy(),
    )
    expected_hybrid_count = len(longitude_labels) * len(latitude_labels)
    if len(hybrid_rows) != expected_hybrid_count:
        raise AssertionError(
            "Hybrid count differs from the eligible Cartesian product."
        )
    if (
        len(longitude_labels) == TUNING_UNIQUE_CANDIDATE_COUNT
        and len(latitude_labels) == TUNING_UNIQUE_CANDIDATE_COUNT
        and expected_hybrid_count != 324
    ):
        raise AssertionError("The complete CatBoost search must create 324 hybrids.")

    hybrid_results = deterministically_rank_hybrids(hybrid_rows)
    winning_hybrid = hybrid_results.iloc[0].to_dict()
    candidate_by_label = {
        item["candidate_label"]: item
        for item in TUNING_CANDIDATES
    }
    unreviewed_winner = {
        "promotion_status": "requires_manual_review",
        "selection_dataset": "validation/X_val",
        "winning_hybrid": winning_hybrid,
        "target_selection": {
            "END_LONG": {
                "candidate_label": winning_hybrid[
                    "longitude_candidate_label"
                ],
                "candidate_params": candidate_by_label[
                    winning_hybrid["longitude_candidate_label"]
                ]["params"].copy(),
                "selected_iterations": records_by_target["END_LONG"][
                    winning_hybrid["longitude_candidate_label"]
                ]["selected_iterations"],
            },
            "END_LAT": {
                "candidate_label": winning_hybrid[
                    "latitude_candidate_label"
                ],
                "candidate_params": candidate_by_label[
                    winning_hybrid["latitude_candidate_label"]
                ]["params"].copy(),
                "selected_iterations": records_by_target["END_LAT"][
                    winning_hybrid["latitude_candidate_label"]
                ]["selected_iterations"],
            },
        },
    }

    target_result_rows = [
        target_result_to_row(records_by_target[target_column][label])
        for target_column, _ in TARGET_SPECS
        for label in [
            item["candidate_label"] for item in TUNING_CANDIDATES
        ]
    ]
    target_results = pd.DataFrame(target_result_rows).sort_values(
        ["target_column", "candidate_index"],
        kind="mergesort",
    ).reset_index(drop=True)
    log_search_summary_to_wandb(
        target_results,
        hybrid_results,
        winning_hybrid,
    )

    display_columns = [
        "candidate_index",
        "candidate_label",
        "target_label",
        "completion_state",
        "eligible_for_hybrid",
        "selected_iterations",
        "best_inner_stop_rmse",
        "validation_rmse",
        "ceiling_reached",
        "truncated_at_ceiling",
    ]
    display(target_results[display_columns])
    display(hybrid_results.head(20))
    display(pd.Series({
        "promotion_status": unreviewed_winner["promotion_status"],
        "selection_dataset": unreviewed_winner["selection_dataset"],
        "longitude_candidate": winning_hybrid[
            "longitude_candidate_label"
        ],
        "latitude_candidate": winning_hybrid[
            "latitude_candidate_label"
        ],
        "validation_haversine_mean_km": winning_hybrid[
            "haversine_mean_km"
        ],
        "validation_haversine_p90_km": winning_hybrid[
            "haversine_p90_km"
        ],
    }))

    predictions_by_target.clear()
    records_by_target.clear()
    del hybrid_rows
    del X_train, y_train, X_val, y_val
    del train_positions, stop_positions
    gc.collect()
    return target_results, hybrid_results, unreviewed_winner


tuning_target_results, hybrid_results, unreviewed_search_winner = (
    run_hyperparameter_tuning()
)


## Train And Validate

After manual promotion, rerun the reviewed target pair on all
`validation/X_train` rows with fixed target-specific iterations. `X_val` is
prediction-only; this mode produces reproducible diagnostics and the
reviewed parameters artifact rather than new model-selection evidence.


In [ ]:
validation_metrics = (
    {}
    if RUN_HYPERPARAMETER_TUNING or RUN_VALIDATION_REFIT
    else reviewed_validation_metrics()
)
target_results = deepcopy(BEST_MODEL_RUN_SUMMARY.get("target_results", {}))
validation_distances = None
execution_status = {
    "hyperparameter_tuning_requested": RUN_HYPERPARAMETER_TUNING,
    "hyperparameter_tuning_completed": bool(
        RUN_HYPERPARAMETER_TUNING and not tuning_target_results.empty
    ),
    "validation_refit_requested": RUN_VALIDATION_REFIT,
    "validation_refit_completed": False,
    "final_fit_requested": RUN_FINAL_FIT,
    "final_iteration_calibration_requested": RUN_FINAL_FIT,
    "final_iteration_calibration_completed": False,
    "final_fit_completed": False,
    "model_bundle_written": False,
    "model_serialized": False,
    "submission_written": False,
}

if RUN_VALIDATION_REFIT:
    X_train, y_train, X_val, y_val = load_cached_validation_data()
    train_pool = build_feature_pool(X_train)
    validation_pool = build_feature_pool(X_val)
    validation_models, target_results = fit_pair_with_fixed_iterations(
        X_train,
        y_train,
        BEST_TARGET_MODEL_PARAMS,
        BEST_TARGET_ITERATIONS,
    )
    train_predictions = combine_target_predictions({
        target_column: validation_models[target_column].predict(train_pool)
        for target_column in TARGET_COLUMNS
    })
    val_predictions = combine_target_predictions({
        target_column: validation_models[target_column].predict(
            validation_pool
        )
        for target_column in TARGET_COLUMNS
    })
    train_metrics, _ = evaluate_long_lat_predictions(
        y_train[TARGET_COLUMNS].to_numpy(),
        train_predictions,
        "train",
    )
    val_metrics, validation_distances = evaluate_long_lat_predictions(
        y_val[TARGET_COLUMNS].to_numpy(),
        val_predictions,
        "val",
    )
    validation_metrics = {
        **train_metrics,
        **val_metrics,
        "generalization/haversine_gap_km": float(
            val_metrics["val/haversine_mean_km"]
            - train_metrics["train/haversine_mean_km"]
        ),
        "fit_seconds": float(
            sum(result["fit_seconds"] for result in target_results.values())
        ),
        "validation_metrics_source": "selected_model_validation_refit",
    }
    execution_status["validation_refit_completed"] = True
    save_best_params_artifact(
        validation_metrics,
        target_results,
        execution_status,
    )
    validation_models.clear()
    del train_pool, validation_pool
    gc.collect()
    display(pd.Series({
        key: value
        for key, value in validation_metrics.items()
        if not key.endswith("seconds")
    }))
elif not RUN_HYPERPARAMETER_TUNING:
    print("Using reviewed validation metrics from BEST_MODEL_RUN_SUMMARY.")
    display(pd.Series(validation_metrics))


## Train vs Validation Plot

Display only the standard mean-Haversine comparison and validation-error histogram.


In [ ]:
if RUN_VALIDATION_REFIT:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    mean_values = [
        validation_metrics["train/haversine_mean_km"],
        validation_metrics["val/haversine_mean_km"],
    ]
    bars = axes[0].bar(
        ["Train", "Validation"],
        mean_values,
        color=["#4C78A8", "#F58518"],
    )
    axes[0].set_ylabel("Mean Haversine distance (km)")
    axes[0].set_title(f"{MODEL_NAME} train vs validation error")
    axes[0].bar_label(bars, fmt="%.3f km")

    axes[1].hist(validation_distances, bins=40, color="#F58518")
    axes[1].axvline(
        validation_distances.mean(),
        color="red",
        linestyle="--",
        label=f"Mean: {validation_distances.mean():.3f} km",
    )
    axes[1].set_xlabel("Haversine distance (km)")
    axes[1].set_ylabel("Validation trips")
    axes[1].set_title(f"{MODEL_NAME} validation error distribution")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    for name in [
        "train_predictions",
        "val_predictions",
        "validation_distances",
        "X_train",
        "y_train",
        "X_val",
        "y_val",
    ]:
        globals().pop(name, None)
    gc.collect()
else:
    print("Skipping validation plots because RUN_VALIDATION_REFIT=False.")


## Final Fit And Kaggle Submission

Recalibrate only the reviewed targets' iterations on a temporary grouped
split of the complete final training matrix, using the shared 20,000-iteration
ceiling and standard 100-iteration early-stopping patience. Discard those
calibration models, refit fresh models on every labeled row, store native
CBM payloads, and generate the competition submission.


In [ ]:
if RUN_FINAL_FIT:
    (
        X_final_train,
        y_final_train,
        X_final_test,
        final_gps_scaler,
        final_categorical_encoder,
    ) = load_cached_final_data()

    (
        calibration_train_positions,
        calibration_stop_positions,
        calibration_groups,
    ) = make_group_disjoint_split(
        X_final_train.index,
        stop_fraction=FINAL_ITERATION_STOP_FRACTION,
        seed=FINAL_ITERATION_CALIBRATION_SEED,
    )
    calibration_train_group_count = len(set(
        calibration_groups[calibration_train_positions]
    ))
    calibration_stop_group_count = len(set(
        calibration_groups[calibration_stop_positions]
    ))
    del calibration_groups
    gc.collect()

    final_iteration_calibration = {}
    final_target_iterations = {}
    for target_column, target_label in TARGET_SPECS:
        full_training_pool = build_labeled_pool(
            X_final_train,
            y_final_train,
            target_column,
        )
        calibration_training_pool = full_training_pool.slice(
            calibration_train_positions
        )
        calibration_stop_pool = full_training_pool.slice(
            calibration_stop_positions
        )
        try:
            calibration_result = fit_target_with_early_stopping(
                BEST_TARGET_MODEL_PARAMS[target_column],
                target_column,
                target_label,
                calibration_training_pool,
                calibration_stop_pool,
                run=None,
            )
            calibration_result.update({
                "calibration_stop_fraction": (
                    FINAL_ITERATION_STOP_FRACTION
                ),
                "calibration_seed": FINAL_ITERATION_CALIBRATION_SEED,
                "calibration_train_rows": len(
                    calibration_train_positions
                ),
                "calibration_stop_rows": len(
                    calibration_stop_positions
                ),
                "calibration_train_trip_groups": (
                    calibration_train_group_count
                ),
                "calibration_stop_trip_groups": (
                    calibration_stop_group_count
                ),
                "overlapping_trip_groups": 0,
            })
            final_iteration_calibration[target_column] = calibration_result
            if calibration_result["completion_state"] != "completed":
                raise RuntimeError(
                    f"Final iteration calibration for {target_column} "
                    "reached the 20,000-iteration ceiling without "
                    "completing early-stopping patience."
                )
            final_target_iterations[target_column] = int(
                calibration_result["selected_iterations"]
            )
        finally:
            del full_training_pool
            del calibration_training_pool, calibration_stop_pool
            gc.collect()

    execution_status["final_iteration_calibration_completed"] = True
    del calibration_train_positions, calibration_stop_positions
    gc.collect()

    final_test_pool = build_feature_pool(X_final_test)
    final_target_results = {}
    model_payloads = {}
    test_predictions_by_target = {}
    try:
        for target_column, target_label in TARGET_SPECS:
            full_training_pool = build_labeled_pool(
                X_final_train,
                y_final_train,
                target_column,
            )
            try:
                selected_iterations = final_target_iterations[
                    target_column
                ]
                final_model = build_model(
                    BEST_TARGET_MODEL_PARAMS[target_column],
                    selected_iterations,
                )
                try:
                    fit_started = time.time()
                    final_model.fit(full_training_pool, verbose=False)
                    fit_seconds = float(time.time() - fit_started)
                    test_prediction = np.asarray(
                        final_model.predict(final_test_pool),
                        dtype=np.float32,
                    )
                    payload = serialize_catboost_model_to_cbm(final_model)
                    verify_cbm_payload_predictions(
                        payload,
                        final_test_pool,
                        test_prediction,
                    )
                finally:
                    del final_model
                    gc.collect()

                test_predictions_by_target[target_column] = test_prediction
                model_payloads[target_label] = {
                    "target_column": target_column,
                    "format": "cbm",
                    "payload": payload,
                }
                final_target_results[target_column] = {
                    "target_column": target_column,
                    "target_label": target_label,
                    "completed": True,
                    "candidate_params": BEST_TARGET_MODEL_PARAMS[
                        target_column
                    ].copy(),
                    "selected_iterations": selected_iterations,
                    "fit_seconds": fit_seconds,
                    "effective_model_params": effective_model_params(
                        BEST_TARGET_MODEL_PARAMS[target_column],
                        selected_iterations,
                    ),
                }
            finally:
                del full_training_pool
                gc.collect()

        combined_test_predictions = combine_target_predictions(
            test_predictions_by_target
        )
        submission_path = SUBMISSION_DIR / SUBMISSION_FILE
        submission = write_submission(
            X_final_test.index,
            long_lat_to_lat_long(combined_test_predictions),
            submission_path,
        )
        execution_status["submission_written"] = True
        execution_status["final_fit_completed"] = True

        bundle_execution_status = {
            **execution_status,
            "model_bundle_written": True,
            "model_serialized": True,
        }
        model_bundle = build_final_model_bundle(
            model_payloads=model_payloads,
            target_results=final_target_results,
            gps_scaler=final_gps_scaler,
            categorical_encoder=final_categorical_encoder,
            feature_columns=X_final_train.columns,
            validation_metrics=validation_metrics,
            final_target_iterations=final_target_iterations,
            final_iteration_calibration=final_iteration_calibration,
            execution_status=bundle_execution_status,
        )
        joblib.dump(model_bundle, MODEL_BUNDLE_PATH)
        execution_status.update({
            "model_bundle_written": True,
            "model_serialized": True,
        })
        save_best_params_artifact(
            validation_metrics,
            final_target_results,
            execution_status,
            final_target_iterations=final_target_iterations,
            final_iteration_calibration=final_iteration_calibration,
        )

        print("Model bundle written to:", MODEL_BUNDLE_PATH.resolve())
        print("Submission written to:", submission_path.resolve())
        display(submission.head())
    finally:
        del final_test_pool
        gc.collect()
else:
    print("Skipping final fit because RUN_FINAL_FIT=False.")
